In [1]:
import cv2
from ultralytics import YOLO
import random

In [17]:
video_path_1 = r"E:\Exam  Cheating detection\Data\Solo\v2\phone_use.mp4"
video_path_2 = r"E:\Exam  Cheating detection\Data\Group\v2\DSC_0205.MOV"
video_path_3 = r"E:\Exam  Cheating detection\Data\Group\v5\DSC_0079(phone).MOV"
video_path_4 = r"E:\Exam  Cheating detection\Data\Group\v5\DSC_0082.MOV"
img_path = r"E:\Exam  Cheating detection\Data\Solo\images_train\1_img002.png"

model_path = r"E:\Exam  Cheating detection\models\best11m.pt"

# YOLO

In [ ]:
def phone_detection(model, input_video_path, show_video=True, save_video=False, output_video_path="output_video.mp4"):
    # Open the input video file
    cap = cv2.VideoCapture(input_video_path)
    #cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        raise Exception("Error: Could not open video file.")

    # Get input video frame rate and dimensions
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the output video writer
    if save_video:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        
        results = model.track(
            frame, 
            iou=0.4, 
            conf=0.1, 
            persist=True, 
            imgsz=1280, # 608 for phone 960, 1280
            verbose=False, 
                #tracker="botsort.yaml"
            ) #bytetrack.yaml
    
        

        if results[0].boxes.id != None: # this will ensure that id is not None -> exist tracks
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, id in zip(boxes, ids):
                # Generate a random color for each object based on its ID
                random.seed(int(id))
                color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
                
                cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3],), color, 2)
                cv2.putText(frame, f"Id {id}", (box[0], box[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
        
        if save_video:
            out.write(frame)

        if show_video:
            frame = cv2.resize(frame, (0, 0), fx=0.7, fy=0.7) #(frame, (0, 0), fx=0.75, fy=0.75)
            cv2.imshow("frame", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Release the input video capture and output video writer
    cap.release()
    if save_video:
        out.release()

    # Close all OpenCV windows
    cv2.destroyAllWindows()

# Example usage:
model = YOLO(model_path)
model.fuse()
model.to('cuda')
phone_detection(model, video_path_3, show_video=True, save_video=False, output_video_path="output_video.mp4")


YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
